In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path
import json, glob, shutil, subprocess, os
import requests
import pandas as pd
import numpy as np
from datasets import load_dataset
SAVE_PATH_ADDRESS = '/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/CovidET-Appraisals/repo'

SAVE_PATH = Path(SAVE_PATH_ADDRESS)

In [5]:
# COVIDET_REPO = 'https://github.com/honglizhan/CovidET-Appraisals-Public.git'

# subprocess.run(["git", "clone", COVIDET_REPO, str(SAVE_PATH)], check=True)

# print("Downloaded repo to:", SAVE_PATH)

In [6]:
all_files = []
for root, dirs, files in os.walk(SAVE_PATH):
    for f in files:
        path = Path(root) / f
        all_files.append(path)

print("Number of files:", len(all_files))
for p in all_files[:100]:
    print(p.relative_to(SAVE_PATH))

Number of files: 96
.gitattributes
README.md
.git/description
.git/packed-refs
.git/HEAD
.git/config
.git/index
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-commit.sample
.git/hooks/pre-push.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-rebase.sample
.git/hooks/applypatch-msg.sample
.git/hooks/pre-receive.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/update.sample
.git/hooks/push-to-checkout.sample
.git/hooks/commit-msg.sample
.git/hooks/prepare-commit-msg.sample
.git/hooks/post-update.sample
.git/info/exclude
.git/refs/heads/main
.git/refs/remotes/origin/HEAD
.git/objects/pack/pack-49992f360d8ebcac5786279627129e0cf8d48d64.pack
.git/objects/pack/pack-49992f360d8ebcac5786279627129e0cf8d48d64.idx
.git/logs/HEAD
.git/logs/refs/remotes/origin/HEAD
.git/logs/refs/heads/main
Human_EVAL/EVAL_analysis.ipynb
Human_EVAL/EVAL_inter-rator_agreement.ipynb
Human_EVAL/TURKERS-EVAL_data-intersection-ALL_6_models.csv
Human_EVAL/TURKERS-EVAL_data-intersection-TOP_3_models.csv
LLM_

In [7]:
candidate_files = []
for ext in ["*.csv", "*.tsv", "*.json", "*.jsonl", "*.xlsx", "*.pkl"]:
    candidate_files.extend(SAVE_PATH.rglob(ext))

print("Candidate data files:")
for p in candidate_files:
    print("-", p.relative_to(SAVE_PATH))

Candidate data files:
- Human_EVAL/TURKERS-EVAL_data-intersection-ALL_6_models.csv
- Human_EVAL/TURKERS-EVAL_data-intersection-TOP_3_models.csv
- data/CovidET-Appraisals.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-1.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-2.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-3.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-4.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-5.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-1.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-2.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-3.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-4.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-5.csv
- LLM_responses/Responses-Cleaned/chatgpt/chatgpt_seed-1.csv
- LLM_responses/Responses-Cleaned/chatgpt/chatgpt_seed-2.csv
- LLM_responses/Response

In [8]:
appraisal_df = pd.read_csv(SAVE_PATH / 'data/CovidET-Appraisals.csv')
print(appraisal_df.head())

                           HIT ID                   Assignment ID  \
0  360ZO6N6KAUVP7FXL2X2G8VG6P9M9W  3QEMNNSB27AQUEJLJE5G86CWSFZ7D6   
1  360ZO6N6KAUVP7FXL2X2G8VG6P9M9W  3PM8NZGV88REY2TH6DFY24WJDNCXQ5   
2  3Z33IC0JD9XEFU96ZUYT34ETXILV9M  33IZTU6J8BCQBI8UYH43X4BKVBSSXK   
3  3Z33IC0JD9XEFU96ZUYT34ETXILV9M  3W8CV64QJCABKDCGK7MQDNI0FXV9H8   
4  3XJOUITW9325U1M3B190OHGXAX9TQQ  3RKNTXVS3W9VDKYQX6G7FTKGGAAA4L   

        Worker ID Reddit ID  \
0   AM6H40LNWSFYA    o6lpwn   
1  A23ARVB31O6LE3    o6lpwn   
2  A23ARVB31O6LE3    o77vmk   
3   AM6H40LNWSFYA    o77vmk   
4  A23ARVB31O6LE3    o78h66   

                                         Reddit Post  dim1  \
0  I dont even know how to speak of this grief. I...   6.0   
1  I dont even know how to speak of this grief. I...   2.0   
2  I'm being vague as to not give away my employe...   1.0   
3  I'm being vague as to not give away my employe...   2.0   
4  My mental health was never great before the pa...   3.0   

                         

In [9]:
import pandas as pd

covid_df = pd.read_csv(SAVE_PATH / 'data/CovidET-Appraisals.csv')

print("Shape:", covid_df.shape)
print("Unique Reddit posts:", covid_df["Reddit ID"].nunique())
print("Unique annotators:", covid_df["Worker ID"].nunique())
print(covid_df["Worker ID"].value_counts())

rating_cols = [c for c in covid_df.columns if c.startswith("dim") and not c.endswith("_rationale")]
rationale_cols = [c for c in covid_df.columns if c.endswith("_rationale")]

print("Rating columns:", len(rating_cols))
print("Rationale columns:", len(rationale_cols))

all_ratings = pd.concat([covid_df[c] for c in rating_cols])
print("Rating min:", all_ratings.dropna().min())
print("Rating max:", all_ratings.dropna().max())
print("Unique rating values:", sorted(all_ratings.dropna().unique()))
print("Missing ratings:", all_ratings.isna().sum())

Shape: (281, 53)
Unique Reddit posts: 241
Unique annotators: 2
Worker ID
A23ARVB31O6LE3    143
AM6H40LNWSFYA     138
Name: count, dtype: int64
Rating columns: 24
Rationale columns: 24
Rating min: 1.0
Rating max: 9.0
Unique rating values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Missing ratings: 1052


In [10]:
covid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 53 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   HIT ID           281 non-null    object 
 1   Assignment ID    281 non-null    object 
 2   Worker ID        281 non-null    object 
 3   Reddit ID        281 non-null    object 
 4   Reddit Post      281 non-null    object 
 5   dim1             277 non-null    float64
 6   dim1_rationale   281 non-null    object 
 7   dim2             266 non-null    float64
 8   dim2_rationale   281 non-null    object 
 9   dim3             273 non-null    float64
 10  dim3_rationale   281 non-null    object 
 11  dim4             278 non-null    float64
 12  dim4_rationale   281 non-null    object 
 13  dim5             260 non-null    float64
 14  dim5_rationale   281 non-null    object 
 15  dim6             271 non-null    float64
 16  dim6_rationale   281 non-null    object 
 17  dim7            

In [11]:
covid_df.describe()

,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,...,dim15,dim16,dim17,dim18,dim19,dim20,dim21,dim22,dim23,dim24
count,277.000000,266.000000,273.000000,278.000000,260.000000,271.000000,261.000000,273.000000,260.000000,276.000000,...,268.000000,44.000000,238.000000,69.000000,270.000000,257.000000,249.000000,268.000000,41.000000,251.000000
mean,2.808664,5.259398,7.249084,3.964029,7.926923,6.391144,3.739464,3.516484,4.973077,7.155797,...,2.570896,2.681818,4.260504,3.521739,4.540741,3.844358,6.698795,7.220149,2.634146,3.338645
std,2.077187,2.780730,1.871872,2.139233,0.969838,2.169262,2.246607,2.398051,2.751491,2.055857,...,1.704426,1.325470,1.625027,1.441085,2.903050,2.223608,1.573994,1.540965,1.996949,2.051551
min,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,7.000000,3.000000,7.000000,6.000000,2.000000,1.000000,2.000000,7.000000,...,1.000000,1.000000,3.000000,3.000000,1.000000,1.000000,6.000000,6.000000,1.000000,1.000000
50%,2.000000,6.000000,8.000000,3.000000,8.000000,7.000000,3.000000,3.000000,6.000000,8.000000,...,2.000000,3.000000,4.000000,4.000000,6.000000,4.000000,7.000000,7.000000,2.000000,3.000000
75%,4.000000,7.000000,9.000000,6.000000,9.000000,8.000000,4.000000,6.000000,7.000000,8.000000,...,3.000000,4.000000,5.000000,4.000000,7.000000,6.000000,8.000000,8.000000,4.000000,5.000000
max,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,...,9.000000,5.000000,9.000000,7.000000,9.000000,9.000000,9.000000,9.000000,9.000000,8.000000


In [12]:
covid_df.head()

,HIT ID,Assignment ID,Worker ID,Reddit ID,Reddit Post,dim1,dim1_rationale,dim2,dim2_rationale,dim3,...,dim20,dim20_rationale,dim21,dim21_rationale,dim22,dim22_rationale,dim23,dim23_rationale,dim24,dim24_rationale
0,360ZO6N6KAUVP7FXL2X2G8VG6P9M9W,3QEMNNSB27AQUEJLJE5G86CWSFZ7D6,AM6H40LNWSFYA,o6lpwn,I dont even know how to speak of this grief. I...,6.0,While the narrator doesn't seem to feel respon...,2.0,The narrator doesn't believe that others in th...,7.0,...,2.0,The narrator does not think that their uncle's...,7.0,The narrator expresses that their uncle's deat...,8.0,The narrator thinks that their uncle's death i...,NaN,The narrator doesn't really mention any person...,2.0,The narrator did not expect their uncle to die...
1,360ZO6N6KAUVP7FXL2X2G8VG6P9M9W,3PM8NZGV88REY2TH6DFY24WJDNCXQ5,A23ARVB31O6LE3,o6lpwn,I dont even know how to speak of this grief. I...,2.0,While the narrator understands they did not pl...,2.0,The narrator mentions how they were unable to ...,9.0,...,1.0,The narrator does not mention ever having face...,9.0,The narrator's uncle's death is taking a huge ...,9.0,The narrator is facing extreme challenge and h...,1.0,The narrator seems to have a strong care for t...,1.0,Though they narrator understands the risk and ...
2,3Z33IC0JD9XEFU96ZUYT34ETXILV9M,33IZTU6J8BCQBI8UYH43X4BKVBSSXK,A23ARVB31O6LE3,o77vmk,I'm being vague as to not give away my employe...,1.0,The narrator only mentions other people or for...,9.0,The narrator mentions how their employer is no...,7.0,...,6.0,While the narrator does not find COVID familia...,6.0,The narrator seems to believe the situation is...,8.0,The narrator finds the situation challenging b...,7.0,The narrator mentions getting vaccinated and a...,1.0,The narrator did not seem to expect the situat...
3,3Z33IC0JD9XEFU96ZUYT34ETXILV9M,3W8CV64QJCABKDCGK7MQDNI0FXV9H8,AM6H40LNWSFYA,o77vmk,I'm being vague as to not give away my employe...,2.0,The narrator doesn't think they are responsibl...,8.0,The narrator believes that other people are mo...,3.0,...,6.0,"The narrator mentions that the situation, the ...",7.0,The narrator believes that it will take a lot ...,8.0,The narrator believes that the situation is ve...,NaN,The narrator doesn't mention any personal valu...,4.0,The narrator did not seem to expect this situa...
4,3XJOUITW9325U1M3B190OHGXAX9TQQ,3RKNTXVS3W9VDKYQX6G7FTKGGAAA4L,A23ARVB31O6LE3,o78h66,My mental health was never great before the pa...,3.0,The narrator states that they had preexisting ...,1.0,The narrator mentions they are distant from th...,6.0,...,5.0,The narrator expresses having struggles with m...,8.0,The narrator mentions that they are the only o...,9.0,The narrator expresses a multitude of struggle...,NaN,The narrator does not express whether or not t...,2.0,The narrator describes pre-pandemic mental hea...
